# Applied Linear Algebra in Data Analysis
## Tutorial 04 — Linear Least Squares

**Instructions**
- Cells marked `# YOUR CODE HERE` require you to write code.
- Cells marked `# YOUR ANSWER HERE` require a written explanation in the markdown cell.
- Do **not** use `np.linalg.lstsq`, `np.linalg.solve`, or any high-level solver unless
  explicitly told to. Implement everything from the normal equations.
- Run cells in order.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['figure.dpi'] = 110
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['axes.grid'] = True
rcParams['grid.alpha'] = 0.3

print('Setup complete.')

### Upload data files
Upload the five CSV files provided with this assignment:
`polyfit.csv`, `timeseries.csv`, `trilatdist.csv`, `trilatctrlpos.csv`, `trilatactpos.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select all five CSVs at once

---
## Problem 1 — Polynomial Curve Fitting

We fit a degree-$n$ polynomial $y = \sum_{k=0}^{n}\beta_k x^k$ to data $(x_l, y_l)_{l=1}^m$
by constructing the **Vandermonde matrix**
$$\mathbf{X} = \begin{bmatrix}
1 & x_1 & x_1^2 & \cdots & x_1^n\\
\vdots & & & & \vdots\\
1 & x_m & x_m^2 & \cdots & x_m^n
\end{bmatrix}
\in \mathbb{R}^{m\times(n+1)}$$
and solving $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$.

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
df_poly = pd.read_csv('polyfit.csv')
x_data  = df_poly['x'].to_numpy()
y_data  = df_poly['y'].to_numpy()
m       = len(x_data)

plt.figure(figsize=(6, 3.5))
plt.scatter(x_data, y_data, s=20, color='steelblue', label='data')
plt.xlabel('x');  plt.ylabel('y');  plt.title('Raw data')
plt.tight_layout();  plt.show()

### 1(a) — Fitting error vs. polynomial order

Fit polynomials of orders $n = 0, 1, \ldots, 10$ and compute the fitting error
$e_n = \|\mathbf{X}\hat{\boldsymbol{\beta}} - \mathbf{y}\|_2$ for each.

In [ ]:
def vandermonde(x, n):
    """
    Build the Vandermonde matrix for data vector x and polynomial degree n.
    Returns X of shape (len(x), n+1) where X[i,k] = x[i]**k.
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    raise NotImplementedError
    # ─────────────────────────────────────────────────────────────────────

# Quick sanity check
V = vandermonde(np.array([1.0, 2.0, 3.0]), 2)
expected = np.array([[1,1,1],[1,2,4],[1,3,9]], dtype=float)
assert np.allclose(V, expected), 'vandermonde() is incorrect'
print('vandermonde() passed basic check.')

In [ ]:
def fit_poly(x, y, n):
    """
    Fit a degree-n polynomial to (x, y) via the normal equations.
    Returns beta_hat of shape (n+1,).
    Use np.linalg.inv (do not use lstsq/solve).
    """
    X = vandermonde(x, n)
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    beta_hat = None
    # ─────────────────────────────────────────────────────────────────────
    return beta_hat


def fitting_error(x, y, beta):
    """Return ||X @ beta - y||_2."""
    n = len(beta) - 1
    X = vandermonde(x, n)
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    error = None
    # ─────────────────────────────────────────────────────────────────────
    return error

In [ ]:
# ── Compute errors for n = 0 … 10 (boilerplate) ─────────────────────────────
orders = np.arange(11)
errors = [fitting_error(x_data, y_data, fit_poly(x_data, y_data, n)) for n in orders]

plt.figure(figsize=(6, 3.5))
plt.plot(orders, errors, 'o-', color='steelblue')
plt.xlabel('Polynomial order $n$')
plt.ylabel('Fitting error $e_n$')
plt.title('Fitting error vs. polynomial order')
plt.xticks(orders)
plt.tight_layout();  plt.show()

# Visualise fits for n = 1, 2, 5
x_fine = np.linspace(x_data.min(), x_data.max(), 300)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n in zip(axes, [1, 2, 5]):
    beta = fit_poly(x_data, y_data, n)
    y_fit = vandermonde(x_fine, n) @ beta
    ax.scatter(x_data, y_data, s=10, color='steelblue', label='data')
    ax.plot(x_fine, y_fit, 'r-', lw=1.5, label=f'n={n}')
    ax.set_title(f'Degree {n}  (e={fitting_error(x_data, y_data, beta):.3f})')
    ax.legend(fontsize=8)
plt.tight_layout();  plt.show()

**Q:** Why does $e_n$ decrease monotonically with $n$?  Can $e_n$ ever be exactly zero?

*Your answer:*

### 1(b) — Model validation (train / test split)

A model that perfectly fits the training data may generalise poorly to new data — it
has **overfit** the noise. Split the data into a training set (80 %) and a test set
(20 %) and evaluate the **validation error**:
$$e_n^{\mathrm{val}} = \|\mathbf{X}_{\mathrm{test}}\hat{\boldsymbol{\beta}} - \mathbf{y}_{\mathrm{test}}\|_2$$
where $\hat{\boldsymbol{\beta}}$ is estimated from the *training* set only.

In [ ]:
def train_test_split(x, y, train_frac=0.8, seed=7):
    """
    Randomly split (x, y) into training and test sets.
    Returns (x_train, y_train, x_test, y_test).
    """
    rng = np.random.default_rng(seed)
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    raise NotImplementedError
    # ─────────────────────────────────────────────────────────────────────

x_tr, y_tr, x_te, y_te = train_test_split(x_data, y_data)
print(f'Train: {len(x_tr)} points   Test: {len(x_te)} points')

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Compute train and validation errors for each order n in range(11).
train_errors = []
val_errors   = []
for n in orders:
    pass   # replace with your implementation
# ─────────────────────────────────────────────────────────────────────────────

# Plot (boilerplate)
plt.figure(figsize=(6, 3.5))
plt.plot(orders, train_errors, 'o-', color='steelblue', label='train error')
plt.plot(orders, val_errors,   's--', color='tomato',   label='validation error')
plt.xlabel('Polynomial order $n$');  plt.ylabel('Error')
plt.title('Train vs. validation error')
plt.xticks(orders);  plt.legend()
plt.tight_layout();  plt.show()

**Q:** How does the validation error curve differ from the training error curve?
What is the optimal polynomial order?

*Your answer:*

### 1(c) — Tikhonov (ridge) regularisation

Instead of minimising $\|\mathbf{X}\boldsymbol{\beta} - \mathbf{y}\|^2$, minimise
$$J(\boldsymbol{\beta}) = \|\mathbf{X}\boldsymbol{\beta} - \mathbf{y}\|^2 + \lambda\|\boldsymbol{\beta}\|^2, \quad \lambda \geq 0$$

Fix the polynomial order at $n = 10$ and the full dataset.
Vary $\lambda$ over a logarithmic grid and observe the effect.

In [ ]:
def fit_poly_regularised(x, y, n, lam):
    """
    Fit a degree-n polynomial with Tikhonov regularisation parameter lam.
    Solves: beta_hat = (X^T X + lam I)^{-1} X^T y
    """
    X = vandermonde(x, n)
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    beta_hat = None
    # ─────────────────────────────────────────────────────────────────────
    return beta_hat


n_fixed  = 10
lambdas  = np.logspace(-4, 3, 60)
reg_errors = [fitting_error(x_data, y_data,
                            fit_poly_regularised(x_data, y_data, n_fixed, lam))
              for lam in lambdas]

# Plot error vs lambda (boilerplate)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].semilogx(lambdas, reg_errors, color='steelblue')
axes[0].set_xlabel('$\\lambda$');  axes[0].set_ylabel('Fitting error')
axes[0].set_title('Fitting error vs. $\\lambda$ ($n=10$)')

# Visualise fits for a few lambdas
for lam, col in zip([1e-3, 1, 100], ['tomato','green','purple']):
    beta = fit_poly_regularised(x_data, y_data, n_fixed, lam)
    axes[1].plot(x_fine, vandermonde(x_fine, n_fixed) @ beta,
                 lw=1.4, label=f'$\\lambda$={lam}')
axes[1].scatter(x_data, y_data, s=10, color='steelblue', zorder=5)
axes[1].set_title('Fitted curves for different $\\lambda$')
axes[1].legend(fontsize=8)
plt.tight_layout();  plt.show()

**Q:** Compare $\hat{\boldsymbol{\beta}}$ for very small and very large $\lambda$.
What does $\lambda \to \infty$ imply about the solution?

*Your answer:*

---
## Problem 2 — Time-Series Smoothing

Given a noisy time series $\mathbf{x} \in \mathbb{R}^N$, find a smooth estimate
$\hat{\mathbf{x}}$ by minimising
$$O(\hat{\mathbf{x}}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 + \lambda\,\|\mathbf{D}\hat{\mathbf{x}}\|^2$$
where $\mathbf{D} \in \mathbb{R}^{(N-2)\times N}$ computes second differences:
$(\mathbf{D}\hat{\mathbf{x}})_i = 2\hat{x}_i - \hat{x}_{i-1} - \hat{x}_{i+1}$.

This is a Tikhonov problem with the structured regulariser $\mathbf{D}$, and has the
closed-form solution
$$\hat{\mathbf{x}} = (\mathbf{I} + \lambda\,\mathbf{D}^\top\mathbf{D})^{-1}\mathbf{x}$$

In [ ]:
# ── Load data (boilerplate) ──────────────────────────────────────────────────
df_ts = pd.read_csv('timeseries.csv')
t_arr = df_ts['t'].to_numpy()
x_arr = df_ts['x'].to_numpy()
N     = len(x_arr)

plt.figure(figsize=(9, 3))
plt.plot(t_arr, x_arr, lw=0.8, color='steelblue', label='noisy signal')
plt.xlabel('t');  plt.ylabel('x');  plt.title('Noisy time series')
plt.tight_layout();  plt.show()

In [ ]:
# ── Build the second-difference matrix D (boilerplate) ───────────────────────
def build_D(N):
    """Second-difference matrix of shape (N-2, N)."""
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    D = None
    # ─────────────────────────────────────────────────────────────────────
    return D

D = build_D(N)
print(f'D shape: {D.shape}')

In [ ]:
def smooth(x, D, lam):
    """
    Return the smoothed signal x_hat that minimises
        ||x - x_hat||^2 + lam * ||D x_hat||^2
    Closed-form solution: x_hat = (I + lam * D^T D)^{-1} x
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    x_hat = None
    # ─────────────────────────────────────────────────────────────────────
    return x_hat


# Plot smoothed signals for different lambda (boilerplate)
lambdas_ts = [0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)
for ax, lam in zip(axes.flat, lambdas_ts):
    x_hat = smooth(x_arr, D, lam)
    ax.plot(t_arr, x_arr,  lw=0.7, color='lightsteelblue', label='noisy')
    ax.plot(t_arr, x_hat,  lw=1.8, color='tomato',         label='smoothed')
    ax.set_title(f'$\\lambda = {lam}$')
    ax.legend(fontsize=8)
plt.suptitle('Time-series smoothing for different $\\lambda$', y=1.01)
plt.tight_layout();  plt.show()

**Q:** What does $\lambda$ control? What happens as $\lambda \to 0$ and $\lambda \to \infty$?

*Your answer:*

---
## Problem 3 — Trilateration

A point $P$ moves along an unknown trajectory $\mathbf{x}_n \in \mathbb{R}^2$.
At each time step $n$ we observe its (noisy) distance from $M$ control points
$\mathbf{a}_1, \ldots, \mathbf{a}_M$:
$$d_{ni}^2 \approx \|\mathbf{x}_n - \mathbf{a}_i\|_2^2$$

**Linearisation.** Taking the difference of two squared-distance equations eliminates the
quadratic term in $\mathbf{x}_n$:
$$d_{nj}^2 - d_{ni}^2
  = \|\mathbf{x}_n\|^2 - 2\mathbf{a}_j^\top\mathbf{x}_n + \|\mathbf{a}_j\|^2
  - \|\mathbf{x}_n\|^2 + 2\mathbf{a}_i^\top\mathbf{x}_n - \|\mathbf{a}_i\|^2$$
$$\Rightarrow\quad 2(\mathbf{a}_i - \mathbf{a}_j)^\top\mathbf{x}_n
   = (d_{nj}^2 - \|\mathbf{a}_j\|^2) - (d_{ni}^2 - \|\mathbf{a}_i\|^2)$$

Taking control point 1 as the reference ($i=1$, $j = 2,\ldots,M$) gives $M-1$ linear
equations in $\mathbf{x}_n$.

In [ ]:
# ── Load data (boilerplate) ──────────────────────────────────────────────────
dists    = pd.read_csv('trilatdist.csv').to_numpy()      # (N_time, M)
ctrl_pos = pd.read_csv('trilatctrlpos.csv').to_numpy()   # (M, 2)
true_pos = pd.read_csv('trilatactpos.csv').to_numpy()    # (N_time, 2)

N_time, M = dists.shape
print(f'Time steps: {N_time}   Control points: {M}')

In [ ]:
def build_linear_system(d_n, ctrl):
    """
    Given distance measurements d_n (shape (M,)) at one time step and
    control point positions ctrl (shape (M, 2)), build the linear system
        A @ x = b
    using control point 0 as the reference.

    Returns A of shape (M-1, 2) and b of shape (M-1,).

    Hint: the i-th row (i = 1 ... M-1) of A is  2*(ctrl[0] - ctrl[i]).
          the i-th entry of b is (d_n[i]**2 - norm(ctrl[i])**2)
                                  - (d_n[0]**2 - norm(ctrl[0])**2).
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────
    A = None
    b = None
    # ─────────────────────────────────────────────────────────────────────
    return A, b

In [ ]:
def estimate_trajectory(dists, ctrl):
    """
    Estimate the 2-D position at every time step via least squares.
    Returns est_pos of shape (N_time, 2).
    """
    N = dists.shape[0]
    est_pos = np.zeros((N, 2))
    for n in range(N):
        A, b = build_linear_system(dists[n], ctrl)
        # ── YOUR CODE HERE: solve the LS problem for x_n ─────────────────
        est_pos[n] = None
        # ─────────────────────────────────────────────────────────────────
    return est_pos

est_pos = estimate_trajectory(dists, ctrl_pos)

# ── Plot trajectory (boilerplate) ────────────────────────────────────────────
plt.figure(figsize=(5.5, 5.5))
plt.plot(true_pos[:, 0], true_pos[:, 1],  'k-',  lw=2,   label='true trajectory')
plt.plot(est_pos[:, 0],  est_pos[:, 1],   'r--', lw=1.2, label='LS estimate')
plt.scatter(ctrl_pos[:, 0], ctrl_pos[:, 1], s=15, color='gray',
            zorder=3, label='control points')
plt.xlabel('$x_1$');  plt.ylabel('$x_2$')
plt.title('Trilateration: estimated vs. true trajectory')
plt.legend();  plt.axis('equal');  plt.tight_layout();  plt.show()

rmse = np.sqrt(np.mean(np.sum((est_pos - true_pos)**2, axis=1)))
print(f'Position RMSE: {rmse:.4f}')

**Q:** What is the minimum number of distance measurements needed to estimate
$\mathbf{x}_n \in \mathbb{R}^2$? Why?

*Your answer:*

### 3(b) — Extension: smoothness prior

If $P$ moves slowly, consecutive positions are similar:
$\|\mathbf{x}_n - \mathbf{x}_{n-1}\|$ is small.

We can encode this as a regulariser. Stack all positions into
$\mathbf{z} = [\mathbf{x}_1^\top, \ldots, \mathbf{x}_N^\top]^\top \in \mathbb{R}^{2N}$
and add a penalty $\lambda\|\mathbf{F}\mathbf{z}\|^2$ where $\mathbf{F}$ encodes
first differences between consecutive positions.

**Task:** design $\mathbf{F}$ and implement the regularised trajectory estimator.

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# 1. Build the block linear system (stacking all time steps)
# 2. Build the first-difference matrix F
# 3. Solve the regularised problem for several values of lambda
# 4. Plot the regularised trajectory against the true trajectory and
#    the unregularised estimate
# ─────────────────────────────────────────────────────────────────────────────

**Q:** How does the smoothness prior improve (or hurt) the estimate?
At what parts of the trajectory does it help most?

*Your answer:*

---
## Problem 4 — Weighted Least Squares

The standard least squares objective weights all residuals equally:
$$O(\mathbf{x}) = \|\mathbf{Ax} - \mathbf{b}\|^2 = \sum_{i=1}^{m}(\tilde{\mathbf{a}}_i^\top \mathbf{x} - b_i)^2$$

The **weighted least squares** objective assigns a positive weight $w_i$ to each equation:
$$O_w(\mathbf{x}) = \sum_{i=1}^{m} w_i(\tilde{\mathbf{a}}_i^\top \mathbf{x} - b_i)^2$$

Let $\mathbf{W} = \operatorname{diag}(w_1, \ldots, w_m)$.

**Tasks**
1. Show that $O_w(\mathbf{x}) = (\mathbf{Ax}-\mathbf{b})^\top \mathbf{W} (\mathbf{Ax}-\mathbf{b})$.
2. Derive the weighted normal equations by setting $\nabla_{\mathbf{x}} O_w = \mathbf{0}$.
3. Give the closed-form expression for $\hat{\mathbf{x}}_w$ (assume $\mathbf{A}$ is full rank).

**Your answer**

*Double-click this cell and write your derivation here.*

1. ...
2. ...
3. ...

### Weighted Polynomial Fitting

Four laboratories measure the input-output response of a system, each covering a non-overlapping range of the input variable $x \in [-1, 1]$:

| Lab | $x$ range | Relative reliability |
|-----|-----------|----------------------|
| 1 | $[-1,\,-0.25]$ | 1.5 |
| 2 | $[-0.25,\,0.10]$ | 2.25 |
| 3 | $[0.10,\,0.50]$ | 9 |
| 4 | $[0.50,\,1.00]$ | 1 |

Lab 3 is 6× more reliable than Lab 1; Lab 2 is 1.5× more reliable than Lab 1; Lab 1 is 1.5× more reliable than Lab 4 (reliability values above are relative to Lab 4).

Fit the data using standard least squares and then weighted least squares, where the weights $w_i$ reflect the reliability of the lab that produced measurement $i$.

In [ ]:
# ── Load data (boilerplate) ────────────────────────────────────────────────
df_wls = pd.read_csv(DATA_DIR / "polyfitweighted.csv")
x_wls = df_wls["x"].values
y_wls = df_wls["y"].values
lab   = df_wls["lab"].values   # integer 1–4 identifying the source lab

# Quick scatter: colour by lab
fig, ax = plt.subplots(figsize=(7, 3.5))
for l, colour in zip([1, 2, 3, 4], ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    mask = lab == l
    ax.scatter(x_wls[mask], y_wls[mask], s=18, color=colour, label=f"Lab {l}")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(); ax.set_title("Weighted fitting data")
plt.tight_layout(); plt.show()

In [ ]:
def fit_poly_weighted(x, y, w, n):
    """
    Weighted least squares fit of a degree-n polynomial.

    Parameters
    ----------
    x : array (m,)   input values (already scaled if needed)
    y : array (m,)   output values
    w : array (m,)   non-negative weights (larger = more reliable)
    n : int           polynomial degree

    Returns
    -------
    beta_hat : array (n+1,)   coefficient vector [β₀, β₁, …, βₙ]

    Hint: form W = diag(w), then solve the weighted normal equations
          (XᵀWX) β = XᵀWy  using np.linalg.solve or np.linalg.lstsq.
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────

    raise NotImplementedError
    # ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Scale x for numerical stability ──────────────────────────────────────
x_sc = x_wls / np.max(np.abs(x_wls))

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
# 1. Choose a polynomial order N_POLY by inspecting the scatter plot above.
# 2. Build weight vector w1 from the reliability ratios in the problem statement
#    (one weight per data point, based on its lab).
# 3. Build weight vector w2 for the revised scenario (Lab 3 is 1000× more
#    reliable than the other labs, which are equally unreliable).
# 4. Fit OLS and both WLS models using fit_poly / fit_poly_weighted.

N_POLY = ...   # your choice

w1 = ...   # weight set 1
w2 = ...   # weight set 2

beta_ols  = ...
beta_wls1 = ...
beta_wls2 = ...
# ──────────────────────────────────────────────────────────────────────────

# ── Plot ──────────────────────────────────────────────────────────────────
x_plot = np.linspace(-1, 1, 300)
X_plot = vandermonde(x_plot, N_POLY)

fig, ax = plt.subplots(figsize=(8, 4))
for l, colour in zip([1, 2, 3, 4], ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    mask = lab == l
    ax.scatter(x_sc[mask], y_wls[mask], s=18, color=colour, label=f"Lab {l}")

ax.plot(x_plot, X_plot @ beta_ols,  "k-",  lw=1.8, label="OLS")
ax.plot(x_plot, X_plot @ beta_wls1, "m--", lw=1.8, label="WLS (set 1)")
ax.plot(x_plot, X_plot @ beta_wls2, "c:",  lw=2.2, label="WLS (set 2)")
ax.set_xlabel("x (scaled)"); ax.set_ylabel("y")
ax.legend(ncol=2); ax.set_title("OLS vs WLS fits")
plt.tight_layout(); plt.show()

print("OLS  β̂:", np.round(beta_ols,  4))
print("WLS1 β̂:", np.round(beta_wls1, 4))
print("WLS2 β̂:", np.round(beta_wls2, 4))

**Q1:** Where do the OLS and WLS (set 1) fits differ most visibly, and why?

**Q2:** In WLS set 2, Lab 3 dominates with weight 1000. What does the fit look like in the Lab 3 region ($x \in [0.10, 0.50]$) compared to the rest of the range?

**Q3:** Which fit would you trust most as a representation of the true underlying function, and why?